# Exercises XP: Text Preprocessing, NER, POS, and Word2Vec

Use this guided notebook to follow the platform instructions step by step. Prefilled cells are ready to run; cells marked TODO expect your code or analysis.

## What you will learn
- Clean and normalize raw reviews with tokenization, stopword removal, and lemmatization.
- Extract linguistic features with named entity recognition (NER) and part-of-speech (POS) tagging.
- Train a simple Word2Vec model and interpret its vector dimensions.
- Visualize word embeddings to reason about semantic neighborhoods.

## What you will create
- A `preprocess_text` function that lowercases, strips punctuation, removes stopwords, and lemmatizes.
- `perform_ner` and `perform_pos_tagging` helpers to analyze raw vs cleaned text.
- A Word2Vec model plus a helper to plot embeddings for inspection.

> Learning point
> Run the setup cells once, then progress through each exercise sequentially. Print intermediate results to verify every helper works before moving on.

## Setup · install libraries
Run once to install spaCy, nltk, gensim, and plotting utilities.

In [ ]:
%pip install --quiet spacy nltk gensim matplotlib seaborn --upgrade

In [ ]:
import nltk
from spacy.cli import download as spacy_download
import spacy

resources = [
    "punkt",
    "punkt_tab",
    "wordnet",
    "omw-1.4",
    "stopwords",
    "averaged_perceptron_tagger",
    "averaged_perceptron_tagger_eng",
    "tagsets",
]
for res in resources:
    nltk.download(res, quiet=True)

spacy_download("en_core_web_sm")

nlp = spacy.load("en_core_web_sm")
print("spaCy pipeline:", nlp.pipe_names)

## Exercise 1 · Explore text preprocessing, NER, and POS tags

Here is the dataset you will reuse in every step.

In [ ]:
data = {
    'Review': [
        "At McDonald's the food was ok and the service was bad.",
        "I would not recommend this Japanese restaurant to anyone.",
        "I loved this restaurant when I traveled to Thailand last summer.",
        "The menu of Loving has a wide variety of options.",
        "The staff was friendly and helpful at Google's employees restaurant.",
        "The ambiance at Bella Italia is amazing, and the pasta dishes are delicious.",
        "I had a terrible experience at Pizza Hut. The pizza was burnt, and the service was slow.",
        "The sushi at Sushi Express is always fresh and flavorful.",
        "The steakhouse on Main Street has a cozy atmosphere and excellent steaks.",
        "The dessert selection at Sweet Treats is to die for!"
    ]
}
raw_reviews = data['Review']
raw_reviews

### 1.1 Build `preprocess_text()`
Create a function that:
1. Lowercases and tokenizes text.
2. Removes punctuation tokens.
3. Removes English stopwords.
4. Applies a lemmatizer.
5. Returns the cleaned string joined by spaces.

Print the processed reviews to confirm every stage works.

In [ ]:
import re
import string
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()


def preprocess_text(text: str) -> str:
    """Lowercase, tokenize, strip punctuation, drop stopwords, and lemmatize a review."""
    # 1. Tokenize the text (lowercased first)
    tokens = word_tokenize(text.lower())

    # 2. Remove punctuation tokens
    tokens = [token for token in tokens if token not in string.punctuation]

    # 3. Filter out stopwords
    tokens = [token for token in tokens if token not in stop_words]

    # 4. Lemmatize remaining tokens
    tokens = [lemmatizer.lemmatize(token) for token in tokens]

    # 5. Join back into a single cleaned string
    return " ".join(tokens)


# Test the function on the first review
print("Test preprocess_text:")
print(preprocess_text(raw_reviews[0]))

### 1.2 Create a cleaned dataset
Apply `preprocess_text` to every review and keep both raw and cleaned versions side by side.

In [ ]:
# Apply preprocess_text to every review
cleaned_reviews = [preprocess_text(review) for review in raw_reviews]

if cleaned_reviews is None:
    raise ValueError("Set cleaned_reviews by applying preprocess_text to raw_reviews.")

for raw, cleaned in zip(raw_reviews, cleaned_reviews):
    print(f"RAW: {raw}")
    print(f"CLEANED: {cleaned}\n")

### 1.3 Named Entity Recognition (NER)
Create `perform_ner(text)` that returns `(entity, label_)` pairs using `en_core_web_sm`. Test it on a few reviews.

In [ ]:
def perform_ner(text: str):
    """Return (entity, label) pairs found by spaCy."""
    # Run the spaCy pipeline on the text
    doc = nlp(text)

    # Collect each entity text and its label_
    entities = [(ent.text, ent.label_) for ent in doc.ents]
    return entities


# Test on the first 3 raw reviews
print("NER test on raw reviews:")
for review in raw_reviews[:3]:
    print(f"Review: {review}")
    print(f"Entities: {perform_ner(review)}\n")

### 1.4 Part-of-Speech tagging (POS)
Create `perform_pos_tagging(text)` using `nltk.pos_tag`. Test it on both raw and cleaned text.

Use `nltk.help.upenn_tagset('NN')` to recall tag meanings if needed.

In [ ]:
from nltk import pos_tag, word_tokenize


def perform_pos_tagging(text: str):
    """Return POS tags for a given text."""
    # Tokenize the text
    tokens = word_tokenize(text)

    # Call nltk.pos_tag on the tokens
    pos_tags = pos_tag(tokens)
    return pos_tags


# Test on first raw review
print("POS test on raw review:")
print(perform_pos_tagging(raw_reviews[0]))

print("\nPOS test on cleaned review:")
print(perform_pos_tagging(cleaned_reviews[0]))

# Example of how to get tag meaning
print("\nMeaning of NN tag:")
nltk.help.upenn_tagset('NN')

### 1.5 Apply NER and POS on raw vs cleaned text
Compare outputs on the same entries to see how preprocessing affects tagging.

In [ ]:
sample_texts = raw_reviews[:2]

print("=" * 60)
print("NER on raw text")
print("=" * 60)
for i, review in enumerate(sample_texts):
    print(f"Review {i+1}: {review}")
    print(f"  Entities: {perform_ner(review)}\n")

print("=" * 60)
print("NER on cleaned text")
print("=" * 60)
for i, review in enumerate(cleaned_reviews[:2]):
    print(f"Review {i+1}: {review}")
    print(f"  Entities: {perform_ner(review)}\n")

print("=" * 60)
print("POS tags on raw text")
print("=" * 60)
for i, review in enumerate(sample_texts):
    print(f"Review {i+1}: {review}")
    print(f"  POS tags: {perform_pos_tagging(review)}\n")

print("=" * 60)
print("POS tags on cleaned text")
print("=" * 60)
for i, review in enumerate(cleaned_reviews[:2]):
    print(f"Review {i+1}: {review}")
    print(f"  POS tags: {perform_pos_tagging(review)}\n")

## Exercise 2 · Plotting word embeddings

### 2.1 Train a Word2Vec model
Vectorize the preprocessed/tokenized dataset with `Word2Vec` from `gensim.models`. Reuse the cleaned text and adjust parameters like `vector_size`, `window`, and `sg`.

In [ ]:
from gensim.models import Word2Vec

# Tokenize on whitespace after preprocessing
tokenized_reviews = [review.split() for review in cleaned_reviews]

print("Example tokenized review:")
print(tokenized_reviews[0])

# Train the Word2Vec model
# vector_size: dimensionality of the word vectors
# window: maximum distance between current and predicted word
# min_count: ignores words with total frequency lower than this
# sg: 0 = CBOW, 1 = Skip-Gram
# epochs: number of training passes over the corpus
w2v_model = Word2Vec(
    sentences=tokenized_reviews,
    vector_size=100,
    window=5,
    min_count=1,
    sg=0,
    epochs=100
)

print("\nWord2Vec model trained successfully!")
w2v_model

### 2.2 Inspect embedding dimensions
Print and interpret the vector size and vocabulary size from the fitted model.

In [ ]:
# Print the vector size and vocabulary size of the model
vector_size = w2v_model.vector_size
vocab_size = len(w2v_model.wv)

print(f"Vector size (dimensions): {vector_size}")
print(f"Vocabulary size: {vocab_size} unique words")
print(f"\nEach word is represented as a dense vector of {vector_size} floating-point numbers.")
print("These dimensions capture semantic relationships between words.")
print("Words used in similar contexts will have similar vector representations.")

# Show the vector for a sample word
sample_word = list(w2v_model.wv.key_to_index.keys())[0]
print(f"\nExample — vector for '{sample_word}' (first 10 dims):")
print(w2v_model.wv[sample_word][:10])

### 2.3 Plot word embeddings
Complete `plot_word_embeddings(model)` to scatter-plot the first two dimensions of the learned vectors and annotate each point with its word. Discuss whether related words cluster together.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


def plot_word_embeddings(model, words=None):
    """
    Plot word embeddings using the first two vector dimensions.

    Parameters
    ----------
    model : gensim Word2Vec model
    words : list of str, optional
        Subset of words to plot. Defaults to all vocabulary words.
    """
    # Use all vocabulary words if none specified
    if words is None:
        words = list(model.wv.key_to_index.keys())

    # Collect vectors for the selected words
    vectors = np.array([model.wv[word] for word in words])

    # Use only the first 2 dimensions for plotting
    x = vectors[:, 0]
    y = vectors[:, 1]

    # Create the figure
    fig, ax = plt.subplots(figsize=(14, 10))

    # Scatter plot
    ax.scatter(x, y, alpha=0.7, s=60, color="steelblue", edgecolors="white", linewidths=0.5)

    # Annotate each point with its word
    for word, xi, yi in zip(words, x, y):
        ax.annotate(
            word,
            xy=(xi, yi),
            xytext=(5, 3),
            textcoords="offset points",
            fontsize=9,
            alpha=0.85
        )

    ax.set_title("Word Embeddings — Dimensions 0 vs 1", fontsize=14, fontweight="bold")
    ax.set_xlabel("Dimension 0")
    ax.set_ylabel("Dimension 1")
    ax.grid(True, linestyle="--", alpha=0.4)
    plt.tight_layout()
    plt.show()


# Call the plotting function using the trained model
plot_word_embeddings(w2v_model)

### Analysis: Are related words close to each other?

**Observations from the plot:**
- With only **10 reviews**, the training corpus is very small, which limits the model's ability to learn meaningful semantic relationships.
- Words that appear in similar contexts (e.g., *restaurant*, *food*, *service*) may cluster together, but results will vary due to the small dataset.

**Possible reasons for the output:**
1. **Tiny corpus**: Word2Vec needs large amounts of data to learn robust embeddings. With 10 sentences, vectors are mostly random.
2. **Low co-occurrence**: Words that only appear once cannot be reliably positioned relative to others.
3. **Plotting only 2 dims**: We discard 98 out of 100 dimensions — most semantic info is lost.
4. **`min_count=1`**: Rare words are included but their vectors are poorly trained.

### 2.4 Go further
- Experiment with different preprocessing (e.g., bigrams, stemming vs lemmatization).
- Tune Word2Vec hyperparameters and compare the plots.
- Try dimensionality reduction (PCA/t-SNE) for richer visualizations.

In [ ]:
# Bonus: PCA to reduce 100 dimensions to 2 for a better visualization
from sklearn.decomposition import PCA

words = list(w2v_model.wv.key_to_index.keys())
vectors = np.array([w2v_model.wv[word] for word in words])

# Reduce to 2 dimensions with PCA
pca = PCA(n_components=2)
reduced = pca.fit_transform(vectors)

fig, ax = plt.subplots(figsize=(14, 10))
ax.scatter(reduced[:, 0], reduced[:, 1], alpha=0.7, s=60,
           color="coral", edgecolors="white", linewidths=0.5)

for word, (xi, yi) in zip(words, reduced):
    ax.annotate(word, xy=(xi, yi), xytext=(5, 3),
                textcoords="offset points", fontsize=9, alpha=0.85)

ax.set_title("Word Embeddings — PCA (100D → 2D)", fontsize=14, fontweight="bold")
ax.set_xlabel("Principal Component 1")
ax.set_ylabel("Principal Component 2")
ax.grid(True, linestyle="--", alpha=0.4)

variance = pca.explained_variance_ratio_
print(f"Explained variance: PC1={variance[0]:.2%}, PC2={variance[1]:.2%}, Total={sum(variance):.2%}")

plt.tight_layout()
plt.show()